# Reformat SY Export

### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [57]:
from typing import Tuple

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm  # Progress bar

import climakitae as ck
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
    set_profile_metadata,
    _format_based_on_structure,
    _construct_profile_dataframe,
    _create_simple_dataframe,
    _create_single_wl_multi_sim_dataframe,
    _create_multi_wl_single_sim_dataframe,
    _create_multi_wl_multi_sim_dataframe,
     _stack_profile_data
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import (
    get_data_options,
    get_subsetting_options,
    get_data,
)

import warnings
warnings.filterwarnings("ignore")


from unittest.mock import MagicMock, call, patch
import pytest

## Where is the issue?

In [8]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
simulations = ["sim1"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)


In [37]:
days_in_year = 365
hours_in_year = 8760

In [41]:
# Check for simulation dimension
has_simulation = "simulation" in sample_data.dims
if has_simulation:
    n_simulations = len(sample_data.simulation)
    simulations = sample_data.simulation.values
else:
    n_simulations = 1
    simulations = [None]

# Get all available time_delta data (all 30 years)
hours_per_day = 24
hours_per_year = 8760
total_hours = len(sample_data.time_delta)
n_years = total_hours // hours_per_year

warming_levels = sample_data.warming_level.values

# Create helper function to extract meaningful simulation labels
def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
    """Extract meaningful simulation label from simulation identifier."""
    if sim is None:
        return f"Sim_{sim_idx+1}"

    sim_str = str(sim)
    if "WRF_" in sim_str:
        # Parse simulation name format: WRF_GCM_params_scenario
        # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
        parts = sim_str.split("_")
        if len(parts) >= 4:
            gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
            params = parts[2]  # e.g., r11i1p1f1
            scenario = parts[3]  # e.g., historical+ssp245

            # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
            if "ssp" in scenario:
                ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                ssp = f"ssp{ssp_part}"
            else:
                ssp = "hist"  # fallback for historical-only

            return f"{gcm}-{params}-{ssp}"
        elif len(parts) >= 2:
            # Fallback for shorter format
            return f"{parts[1]}-{sim_idx+1}"
        else:
            return f"Sim_{sim_idx+1}"
    else:
        # Ensure uniqueness by adding index for non-WRF format
        base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
        return f"{base_name}-{sim_idx+1}"

In [39]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}
    #! -start
    profile_data_reshaped = {}
    profile_data_1d = {}
    #! -end

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Reshape to (days_in_year, 24) for the final DataFrame
                profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                    days_in_year, hours_per_day
                )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_reshaped
                #! -start
                profile_data_reshaped[key] = profile_reshaped
                profile_data_1d[key] = profile_1d
                #! -end

                pbar.update(1)

    # # Create the multi-index DataFrame structure
    # df_profile = _construct_profile_dataframe(
    #     profile_data=profile_data,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return profile_data_reshaped, profile_data_1d

In [14]:
profile_data_reshaped, profile_data_1d = compute_profile(sample_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

In [35]:
sub_reshaped = profile_data_reshaped[('WL_1.5',
  'sim1-1')]
print(len(sub_reshaped[1]))
print(len(profile_data_reshaped[('WL_1.5',
  'sim1-1')]))
sub_1d = profile_data_1d[('WL_1.5',
  'sim1-1')]
print(len(sub_1d))

24
365
8760


profile_data_1d is 8760 long, while profile_data_reshaped is a set of 365 arrays, each 24 long. 

In [52]:
def _construct_profile_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    days_in_year: int,
    hours_per_day: int,
) -> pd.DataFrame:
    """
    Construct a DataFrame from profile data based on warming level and simulation dimensions.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (warming_level, simulation) keys and profile arrays as values
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to extract simulation labels
    days_in_year : int
        Number of days in the year (365 or 366)
    hours_per_day : int
        Number of hours per day (24)

    Returns
    -------
    pd.DataFrame
        Structured DataFrame with appropriate column structure based on dimensions
    """
    hours = np.arange(1, 25, 1)
    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)
    print(f"n_warming_levels: {n_warming_levels}")
    print(f"n_simulations: {n_simulations}")

    if n_warming_levels == 1 and n_simulations == 1:
        return _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )
    # elif n_warming_levels == 1 and n_simulations > 1:
    #     return _create_single_wl_multi_sim_dataframe(
    #         profile_data,
    #         warming_levels[0],
    #         simulations,
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )
    # elif n_warming_levels > 1 and n_simulations == 1:
    #     return _create_multi_wl_single_sim_dataframe(
    #         profile_data,
    #         warming_levels,
    #         simulations[0],
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )
    # else:
    #     return _create_multi_wl_multi_sim_dataframe(
    #         profile_data,
    #         warming_levels,
    #         simulations,
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )


In [53]:
# Create the multi-index DataFrame structure
df_profile_reshaped = _construct_profile_dataframe(
    profile_data=profile_data_reshaped,
    warming_levels=warming_levels,
    simulations=simulations,
    sim_label_func=_get_simulation_label,
    days_in_year=days_in_year,
    hours_per_day=hours_per_day,
)

n_warming_levels: 1
n_simulations: 1


In [54]:
profile_data=profile_data_reshaped
warming_levels=warming_levels
simulations=simulations
sim_label_func=_get_simulation_label
days_in_year=days_in_year
hours_per_day=hours_per_day
hours = np.arange(1, 25, 1)

_create_simple_dataframe, and its ilk, will need to be modified.

In [ ]:
def _create_simple_dataframe(
    profile_data: dict,
    warming_level: float,
    simulation: any,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
) -> pd.DataFrame:
    """
    Create a simple DataFrame for single warming level and single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        Simple DataFrame with hour columns
    """

    wl_key = warming_level
    sim_key = sim_label_func(simulation, 0)

    # Create MultiIndex columns
    col_tuples = [(hour, sim_key) for hour in hours]
    multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        hours_per_day=hours_per_day,
        wl_names=[f"WL_{wl_key}"],
        sim_names=[sim_key],
        hour_first=True,
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, days_in_year + 1, 1),
    )


In [ ]:
test_profile = _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
1,0.146083,0.601752,0.643406,0.661426,0.244153,0.186032,0.764742,0.339169,0.130836,0.671624,...,0.787371,0.861354,0.956166,0.927957,0.971481,0.980319,0.373168,0.662459,0.553923,0.835708
2,0.534043,0.176257,0.433815,0.280340,0.504059,0.273410,0.041843,0.856412,0.203384,0.795836,...,0.439537,0.767686,0.245849,0.289890,0.377936,0.789979,0.712086,0.995068,0.758709,0.278893
3,0.554116,0.456551,0.183198,0.746288,0.259266,0.551655,0.573362,0.753642,0.225652,0.507997,...,0.420857,0.059633,0.403470,0.373588,0.520273,0.603695,0.087620,0.106474,0.625969,0.891961
4,0.046726,0.457501,0.857563,0.295488,0.637050,0.808838,0.990992,0.092879,0.719169,0.319264,...,0.845793,0.753270,0.294384,0.256120,0.144049,0.807974,0.559416,0.496401,0.194495,0.967202
5,0.458929,0.215547,0.940435,0.149193,0.301365,0.169436,0.334092,0.797080,0.711530,0.640018,...,0.388964,0.904963,0.086676,0.229268,0.876842,0.258361,0.319169,0.105288,0.078511,0.355849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,0.389141,0.928255,0.104735,0.918974,0.635717,0.927760,0.200696,0.441029,0.673912,0.144756,...,0.440050,0.645648,0.356067,0.600539,0.974754,0.228671,0.180628,0.221722,0.550693,0.002759
362,0.232997,0.813742,0.564652,0.397301,0.325592,0.217124,0.534877,0.134443,0.392834,0.808779,...,0.590870,0.056448,0.654725,0.353059,0.767479,0.055942,0.196400,0.692010,0.598159,0.681935
363,0.383279,0.497739,0.389932,0.852613,0.984696,0.144115,0.430024,0.320462,0.547550,0.149906,...,0.387008,0.781057,0.741202,0.816780,0.362167,0.940617,0.201962,0.072856,0.254083,0.794213


## Testing

In [ ]:
# Set up the Standard Year generator
profile_selections = {  
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    # "warming_level": [warming_level], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"], 
    # "latitude": (latitude - 0.02, latitude + 0.02), 
    # "longitude": (longitude - 0.02, longitude + 0.02),  
    # "cached_area": area_name, 

    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True, 
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

In [ ]:
# the function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)